In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "concept_drift"

# Real Concept Drift via a Joint-vs-Marginal Discriminator

Every other discriminator notebook here (`concept_drift_detection-discriminator.ipynb` and its
MLP/HGB siblings) trains a classifier on `s2_bands` alone to tell two years apart. That is a
two-sample test on the **marginal P(X)**: it detects covariate shift and is blind to a change in
P(Y|X) unless that change also moves X's marginal.
`concept_drift_detection-class-conditional-hgb.ipynb` reaches P(Y|X) indirectly, running one
X-only discriminator per class and reading the asymmetry between them.

This notebook measures it directly. Factor the joint density ratio between period A and B:

$$\frac{P_B(X,Y)}{P_A(X,Y)} = \underbrace{\frac{P_B(X)}{P_A(X)}}_{\text{covariate shift}}
\cdot \underbrace{\frac{P_B(Y|X)}{P_A(Y|X)}}_{\text{real concept drift}}$$

Fit two discriminators of *"which year did this row come from"* on **identical rows**: one on `X`
alone, one on `[X, y]`. The first can only exploit the left factor; the second sees both. The
**separability gap is the real-drift signal**:

```
gap_auc = auc(joint) - auc(marginal)
```

Two constraints make the gap mean that, and both are enforced by construction in
`src/drift/joint.py` rather than left to this notebook:

1. **Matched priors** (Section 3). Disturbance rates run ~1.5%-3.1% across years here. Left
   alone, the joint discriminator wins by simply reading `y`, and reports prior shift under a new
   name. Every side of every comparison is drawn to the same `(n_class1, n_class0)` design, so
   P(y=1) is identical on both sides. Prior shift is genuine drift, but it is a different kind and
   is reported plainly in Section 1 instead of being smuggled into this statistic.

   With priors matched at $\pi^*$, each side's matched conditional depends only on
   $(\pi^*, P(x|y))$ and $\pi^*$ is shared - so the two matched conditionals are equal **iff**
   both class-conditionals are equal. The gap therefore fires on class-*asymmetric* movement of
   P(x|y), which is what a P(Y|X) change is. This is the same criterion the class-conditional
   notebook reads off two separate AUCs, collapsed into one null-calibrated number.

2. **Shared rows.** The two discriminators see the same rows, differing only by the label column.
   Otherwise the gap conflates drift with the luck of two different draws.

**What makes it checkable** (Sections 6 and 7): a two-way injection control on the real S2 bands.
Symmetric label flips inside a feature-space region move P(Y|X) while leaving P(X) and P(y)
untouched - the gap must rise, the marginal must not. Importance resampling on X moves P(X) while
preserving P(Y|X) exactly - the marginal must rise, the gap must not. If either direction fails,
the pair-level readings in Section 9 should not be trusted. The same two controls run on synthetic
data in `tests/drift/test_joint.py`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.drift.features import prepare_features_from_arrays
from src.drift.joint import (
    DEFAULT_HGB_PARAMS,
    carve_pools,
    classify_gap,
    design_sizes,
    draw_disjoint_sides,
    draw_side,
    fit_label_model,
    flip_labels_in_region,
    importance_resample,
    joint_marginal_gap,
    residual_channel,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Paths
DATA_PATH = str(PROJECT_ROOT / "training_data_with_features.zarr")
SPLIT_PATH = str(PROJECT_ROOT / "data_split.npz")

OUTPUT_ROOT = EXPERIMENTS_DIR / "concept_drift_real_joint_all_prev"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PAIR_RESULTS_PATH = OUTPUT_ROOT / "pair_results.pkl"
NULL_RESULTS_PATH = OUTPUT_ROOT / "null_results.pkl"
FLIP_RESULTS_PATH = OUTPUT_ROOT / "flip_injection_results.pkl"
COVARIATE_RESULTS_PATH = OUTPUT_ROOT / "covariate_control_results.pkl"
RESIDUAL_RESULTS_PATH = OUTPUT_ROOT / "residual_results.pkl"
PRIOR_RATE_PATH = OUTPUT_ROOT / "prior_rate_table.csv"
GAP_TABLE_PATH = OUTPUT_ROOT / "gap_table.csv"

SPLIT_NAMES = ["train", "val", "test"]
SPLIT_SEED = {"train": 1, "val": 2, "test": 3}

# --- Design knobs -------------------------------------------------------------------
# class-0 rows per class-1 row in the matched design. The design prior is shared by both
# sides at any ratio, so validity does not depend on this - but a near-balanced pi* is
# where the label column carries the most information and the gap has the most power.
# The cost is that marg_test_auc here is NOT comparable to `x_only_auc` in the
# class-conditional notebook, which uses the natural ~50:1 class mix.
CLASS0_PER_CLASS1 = 4
CLASS0_CAP = 500_000            # absolute safety cap; not binding at the ratio above

# Fraction of each train year held back for fitting the Section 8 label model h. Carved
# out in every section, not just Section 8, so all sections run on identical rows.
H_HOLDOUT_FRACTION = 0.5
H_FIT_CAP = 300_000             # row cap when fitting h (the holdout is millions of rows)

NULL_ELEVATED_Z = 2.0           # "elevated" = this many null-std above the null mean
MIN_AUC_BAND = 0.005            # absolute floor on that band; see elevation_band()
N_NULL_REPEATS = 8              # per year; the dominant cost of the whole notebook
N_ZERO_REPEATS = 4              # repeats of the no-injection point, to band each sweep

# --- Injection knobs ----------------------------------------------------------------
INJECTION_YEAR_PREV = 2018
INJECTION_YEAR_CURR = 2019
INJECTION_BAND = "B8A"          # band whose upper half defines the injection region
INJECTION_REGION_QUANTILE = 0.5
FLIP_FRACTIONS = [0.1, 0.25, 0.5, 0.75, 1.0]
COVARIATE_ALPHAS = [1.0, 2.0, 3.0]
COVARIATE_KEEP_FRACTION = 0.5

HGB_PARAMS = dict(DEFAULT_HGB_PARAMS, random_state=RANDOM_STATE)

print(f"Output directory: {OUTPUT_ROOT.resolve()}")

In [ ]:
ds = xr.open_zarr(DATA_PATH)
splits = np.load(SPLIT_PATH)

split_pixel_indices = {name: splits[f"{name}_pixel_indices"] for name in SPLIT_NAMES}

year_values = ds.year.values
n_years = len(year_values)
band_names = list(ds.s2_band.values)
INJECTION_BAND_IDX = band_names.index(INJECTION_BAND)

print(f"Dataset loaded. Year range: {year_values.min()} - {year_values.max()} ({n_years} years)")
for name in SPLIT_NAMES:
    print(f"  {name}: {len(split_pixel_indices[name]):,} pixels")
print(f"Bands: {band_names}")
print(f"Injection region band: {INJECTION_BAND} (index {INJECTION_BAND_IDX})")

## 1. Prior Shift, Reported Separately

P(y=1) per year and split, straight from `disturbances` - no model involved.

This table is why Section 3 matches priors. A joint discriminator handed unmatched years wins by
reading `y` alone, and the resulting "drift" is just this table in disguise. Prior shift is real
drift; it is simply a *different* kind, and it is measured here directly rather than inferred from
a classifier.

In [ ]:
def compute_prior_rate_table(ds, split_pixel_indices):
    """P(y=1) per year x split, straight from `disturbances`. No model."""
    rows = []
    for split_name, pixel_indices in split_pixel_indices.items():
        dist = ds.isel(pixel=pixel_indices).disturbances.values  # (n_pixels, n_years)
        for year_idx, year_val in enumerate(year_values):
            col = dist[:, year_idx]
            valid = col[np.isin(col, [0, 1])]
            if len(valid) == 0:
                continue
            rows.append({
                "split": split_name,
                "year": int(year_val),
                "n_valid": int(len(valid)),
                "n_pos": int((valid == 1).sum()),
                "positive_rate": float(np.mean(valid == 1)),
            })
    return pd.DataFrame(rows)


prior_rate_df = compute_prior_rate_table(ds, split_pixel_indices)
prior_rate_df.to_csv(PRIOR_RATE_PATH, index=False)
display(prior_rate_df.pivot(index="year", columns="split", values="positive_rate"))

plt.figure(figsize=(9, 4))
for split_name, split_df in prior_rate_df.groupby("split"):
    plt.plot(split_df["year"], split_df["positive_rate"], marker="o", label=split_name)
plt.ylabel("P(y=1)")
plt.xlabel("Year")
plt.title("Disturbance prior rate per year (prior shift - not a model output)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Per-Year Feature Cache

S2 bands only, via `src.drift.features.prepare_features_from_arrays` - the same function the other
drift notebooks use, so the feature definition (including its all-years mean imputation, and that
imputation's known temporal-leakage caveat) is shared rather than re-implemented here.

Year 0 (2016) has no usable rows by that function's contract and drops out; every later year is
usable.

In [ ]:
print("Loading arrays into RAM (one-time cost)...")
load_start = time.time()

feat_cache = {name: {} for name in SPLIT_NAMES}
for split_name in SPLIT_NAMES:
    ds_split = ds.isel(pixel=split_pixel_indices[split_name])
    s2_split = ds_split.s2_bands.values.astype(np.float32)
    dist_split = ds_split.disturbances.values
    for year_idx in range(n_years):
        feat_cache[split_name][year_idx] = prepare_features_from_arrays(
            s2_split, dist_split, year_idx, x_dtype=np.float32, y_dtype=np.uint8
        )
    del s2_split, dist_split
    print(f"  {split_name} cached ({time.time() - load_start:.1f}s elapsed)")

USABLE_YEAR_IDX = [
    idx for idx in range(n_years)
    if all(feat_cache[name][idx][0] is not None for name in SPLIT_NAMES)
]
print(f"\nUsable year indices: {USABLE_YEAR_IDX} -> {[int(year_values[i]) for i in USABLE_YEAR_IDX]}")
print(f"Feature cache built in {time.time() - load_start:.1f}s")

## 3. The Matched Design

Each `(split, year)` is partitioned by `carve_pools`:

- an **h-holdout** (train split only, `H_HOLDOUT_FRACTION`) reserved for the Section 8 label model
  and never shown to any discriminator. It is carved in *every* section, not just Section 8, so
  the `[X, y]` and residual variants run on identical rows and their gaps are comparable.
- **per-class pools** that every comparison draws from.

`design_sizes` then fixes one `(n_class1, n_class0)` per split, used by every pair fit, every null
fit and every injection fit in the notebook. Two consequences worth being explicit about:

- **Priors are matched by construction.** Both sides get the same class counts, so P(y=1) is
  identical on both sides of every comparison, in every section.
- **Row counts are fixed too.** AUC variance and HGB's overfitting behaviour are both strongly
  n-dependent, so a null measured at a different row count would not be a floor for the pair.
  Sizes are halved so a same-year null can draw two *disjoint* sides at the size a real pair uses.

Both are set by the scarcest year, so the design is bounded by the smallest class-1 count across
all years.

In [ ]:
pools = {name: {} for name in SPLIT_NAMES}
for split_name in SPLIT_NAMES:
    holdout_fraction = H_HOLDOUT_FRACTION if split_name == "train" else 0.0
    for year_idx in USABLE_YEAR_IDX:
        _, y_year = feat_cache[split_name][year_idx]
        pools[split_name][year_idx] = carve_pools(
            y_year,
            np.random.default_rng([RANDOM_STATE, SPLIT_SEED[split_name], year_idx]),
            holdout_fraction=holdout_fraction,
        )

DESIGN = {
    split_name: design_sizes(pools[split_name].values(), CLASS0_PER_CLASS1, CLASS0_CAP)
    for split_name in SPLIT_NAMES
}

design_rows = []
for split_name in SPLIT_NAMES:
    n_class1, n_class0 = DESIGN[split_name]
    design_rows.append({
        "split": split_name,
        "n_class1_per_side": n_class1,
        "n_class0_per_side": n_class0,
        "rows_per_side": n_class1 + n_class0,
        "rows_per_pair": 2 * (n_class1 + n_class0),
        "matched_prior": n_class1 / (n_class1 + n_class0),
        "h_holdout_rows": sum(
            len(pools[split_name][i].holdout_class1) + len(pools[split_name][i].holdout_class0)
            for i in USABLE_YEAR_IDX
        ),
    })
design_df = pd.DataFrame(design_rows)
display(design_df)

print(f"Matched prior pi* = {design_df.loc[0, 'matched_prior']:.3f} on both sides of every comparison.")
print("Compare against the natural per-year rates in Section 1 - the design is deliberately")
print("closer to balanced, which is where the label column is most informative.")

## 4. Running One Comparison

`run_pair` draws both sides to the design, then hands them to `joint_marginal_gap`, which fits the
marginal and joint discriminators on the same rows and returns both plus their gap.

Results are checkpointed per pair to a pickled DataFrame written atomically after every row, and
filtered against before any work is done - the resumable pattern used by the other drift
notebooks. Interrupting and re-running picks up where it stopped.

In [ ]:
_DISC_KEYS = [
    "n_train", "n_val", "n_test",
    "val_auc", "test_auc", "val_f1", "test_f1", "val_logloss", "test_logloss",
]
GAP_KEYS = ["gap_auc", "gap_val_auc", "gap_logloss_reduction"]
PAIR_RESULTS_COLUMNS = (
    ["pair", "year_prev", "year_curr"]
    + [f"marg_{k}" for k in _DISC_KEYS]
    + [f"joint_{k}" for k in _DISC_KEYS]
    + GAP_KEYS
    + ["time_sec"]
)


def load_pickle_checkpoint(path, columns):
    if path.exists():
        return pd.read_pickle(path)
    return pd.DataFrame(columns=columns)


def save_pickle_checkpoint(df, path):
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_pickle(tmp_path)
    tmp_path.replace(path)


def adjacent_rows(df):
    """Adjacent-year rows of a results frame.

    Recomputed at each use rather than cached as a boolean mask: the results frames grow
    as later sections append to them, and a mask built against an earlier version no
    longer aligns.
    """
    return df[(df["year_curr"] - df["year_prev"]) == 1]


def elevation_band(null_std):
    """Half-width above a null mean that counts as "elevated".

    Floored at MIN_AUC_BAND for two reasons: a null std estimated from a handful of
    repeats can come out arbitrarily small by luck, and an AUC difference below that
    floor is not actionable however significant it computes. `std` of a single repeat is
    NaN, which is treated as zero rather than propagating.
    """
    null_std = 0.0 if not np.isfinite(null_std) else float(null_std)
    return max(NULL_ELEVATED_Z * null_std, MIN_AUC_BAND)


def draw_pair_sides(idx_prev, idx_curr, seed_tag):
    """Both sides of one year pair, drawn to DESIGN, for all three splits."""
    sides = {}
    for split_name in SPLIT_NAMES:
        rng = np.random.default_rng([RANDOM_STATE, SPLIT_SEED[split_name], *seed_tag])
        n_class1, n_class0 = DESIGN[split_name]
        X_year, y_year = feat_cache[split_name][idx_prev]
        idx_a = draw_side(pools[split_name][idx_prev], n_class1, n_class0, rng)
        X_a, y_a = X_year[idx_a], y_year[idx_a]

        X_year, y_year = feat_cache[split_name][idx_curr]
        idx_b = draw_side(pools[split_name][idx_curr], n_class1, n_class0, rng)
        X_b, y_b = X_year[idx_b], y_year[idx_b]

        sides[split_name] = (X_a, y_a, X_b, y_b)
    return sides


PAIR_RESULTS_DF = load_pickle_checkpoint(PAIR_RESULTS_PATH, PAIR_RESULTS_COLUMNS)
print(f"Loaded {len(PAIR_RESULTS_DF)} cached pair result(s) from {PAIR_RESULTS_PATH}")


def run_pair(idx_prev, idx_curr):
    global PAIR_RESULTS_DF
    year_prev = int(year_values[idx_prev])
    year_curr = int(year_values[idx_curr])
    pair_name = f"{year_prev} vs {year_curr}"

    cache_hit = PAIR_RESULTS_DF[PAIR_RESULTS_DF["pair"] == pair_name]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    start = time.time()
    sides = draw_pair_sides(idx_prev, idx_curr, seed_tag=(idx_prev, idx_curr))
    metrics = joint_marginal_gap(sides, HGB_PARAMS)
    if metrics is None:
        return None

    row = {
        "pair": pair_name,
        "year_prev": year_prev,
        "year_curr": year_curr,
        **metrics,
        "time_sec": float(time.time() - start),
    }
    PAIR_RESULTS_DF = pd.concat([PAIR_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(PAIR_RESULTS_DF, PAIR_RESULTS_PATH)
    return row

## 5. Adjacent-Year Pairs

In [ ]:
adjacent_start = time.time()
adjacent_idx_pairs = list(zip(USABLE_YEAR_IDX[:-1], USABLE_YEAR_IDX[1:]))

for idx_prev, idx_curr in tqdm(adjacent_idx_pairs, desc="Adjacent-year pairs"):
    run_pair(idx_prev, idx_curr)
print(f"Adjacent-year section done in {time.time() - adjacent_start:.1f}s")

display(
    adjacent_rows(PAIR_RESULTS_DF)
    .sort_values("year_curr")[["pair", "marg_test_auc", "joint_test_auc", "gap_auc", "time_sec"]]
    .reset_index(drop=True)
)

## 6. All-Previous-Year Pairs (resumable)

Every `year_curr` against every earlier `year_prev`, matching the "each year vs all previous
years" framing of the other drift notebooks. Already-computed pairs are skipped.

In [ ]:
all_prev_start = time.time()
all_prev_jobs = [
    (idx_prev, idx_curr)
    for position, idx_curr in enumerate(USABLE_YEAR_IDX)
    for idx_prev in USABLE_YEAR_IDX[:position]
]

for idx_prev, idx_curr in tqdm(all_prev_jobs, desc="All-previous-year pairs"):
    run_pair(idx_prev, idx_curr)
print(f"All-previous-year section done in {time.time() - all_prev_start:.1f}s")
print(f"Total cached pair rows: {len(PAIR_RESULTS_DF)}")

## 7. Same-Year Null

For each year, split that year's pools into two **disjoint** halves at the same design size a real
pair uses, and run the identical machinery. True separability is 0.5 and the true gap is 0 by
construction - same year, same design, same prior on both sides. Repeated `N_NULL_REPEATS` times
per year so "elevated" is judged against a distribution rather than one draw.

This is the floor, and it is not optional. With a 200-tree HGB against six-figure row counts,
small pipeline artifacts (imputation edges, chunk boundaries, the label column's finite-sample
leverage) produce a small non-zero gap with no drift present. Real gaps are read against this
floor, never against a nominal 0.

In [ ]:
NULL_RESULTS_COLUMNS = ["year", "repeat_idx"] + [f"marg_{k}" for k in _DISC_KEYS] + \
    [f"joint_{k}" for k in _DISC_KEYS] + GAP_KEYS

NULL_RESULTS_DF = load_pickle_checkpoint(NULL_RESULTS_PATH, NULL_RESULTS_COLUMNS)
print(f"Loaded {len(NULL_RESULTS_DF)} cached null result(s) from {NULL_RESULTS_PATH}")


def run_null(year_idx, repeat_idx):
    global NULL_RESULTS_DF
    year_val = int(year_values[year_idx])

    cache_hit = NULL_RESULTS_DF[
        (NULL_RESULTS_DF["year"] == year_val) & (NULL_RESULTS_DF["repeat_idx"] == repeat_idx)
    ]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    sides = {}
    for split_name in SPLIT_NAMES:
        rng = np.random.default_rng(
            [RANDOM_STATE, SPLIT_SEED[split_name], 991, year_idx, repeat_idx]
        )
        n_class1, n_class0 = DESIGN[split_name]
        idx_a, idx_b = draw_disjoint_sides(pools[split_name][year_idx], n_class1, n_class0, rng)
        X_year, y_year = feat_cache[split_name][year_idx]
        sides[split_name] = (X_year[idx_a], y_year[idx_a], X_year[idx_b], y_year[idx_b])

    metrics = joint_marginal_gap(sides, HGB_PARAMS)
    if metrics is None:
        return None

    row = {"year": year_val, "repeat_idx": repeat_idx, **metrics}
    NULL_RESULTS_DF = pd.concat([NULL_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(NULL_RESULTS_DF, NULL_RESULTS_PATH)
    return row


null_start = time.time()
null_jobs = [(y, r) for y in USABLE_YEAR_IDX for r in range(N_NULL_REPEATS)]
for year_idx, repeat_idx in tqdm(null_jobs, desc="Same-year null"):
    run_null(year_idx, repeat_idx)
print(f"Null calibration done in {time.time() - null_start:.1f}s")

null_summary_df = NULL_RESULTS_DF.groupby("year", as_index=False).agg(
    gap_null_mean=("gap_auc", "mean"),
    gap_null_std=("gap_auc", "std"),
    marg_null_mean=("marg_test_auc", "mean"),
    marg_null_std=("marg_test_auc", "std"),
    n_repeats=("gap_auc", "count"),
)
display(null_summary_df)

plt.figure(figsize=(9, 4))
plt.axhline(0.0, color="black", linewidth=1)
for year_val, year_df in NULL_RESULTS_DF.groupby("year"):
    plt.scatter([year_val] * len(year_df), year_df["gap_auc"], alpha=0.6)
plt.errorbar(
    null_summary_df["year"], null_summary_df["gap_null_mean"],
    yerr=NULL_ELEVATED_Z * null_summary_df["gap_null_std"],
    fmt="_", markersize=25, capsize=5, color="black",
    label=f"null mean +/- {NULL_ELEVATED_Z:g} sd",
)
plt.ylabel("gap_auc under the null")
plt.xlabel("Year")
plt.title("Same-year null: the gap's noise floor")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Control A - Conditional Injection (the gap must fire)

Pick one year pair. Inside a feature-space region (rows above the median of one band in
`year_curr`), swap labels symmetrically: `k` class-1 rows become 0 and `k` class-0 rows become 1.

- **P(X) is untouched** - no row moves, only labels trade places
- **P(y=1) is untouched** - the swap is symmetric, class counts are preserved exactly
- **P(y|x) changes**, inside the region only

That is real concept drift with the covariate and prior channels pinned. As the flip fraction
rises, `gap_auc` must rise with it while `marg_test_auc` stays inside its null band - the marginal
discriminator is looking at byte-identical X on both sides and has nothing to find.

The zero-injection point is repeated `N_ZERO_REPEATS` times to give this sweep its own null band
at its own design, rather than borrowing Section 7's.

In [ ]:
FLIP_RESULTS_COLUMNS = ["year_prev", "year_curr", "injection_fraction", "repeat_idx",
                        "n_flipped_test", "marg_test_auc", "joint_test_auc", "gap_auc"]
FLIP_RESULTS_DF = load_pickle_checkpoint(FLIP_RESULTS_PATH, FLIP_RESULTS_COLUMNS)

idx_inj_prev = int(np.where(year_values == INJECTION_YEAR_PREV)[0][0])
idx_inj_curr = int(np.where(year_values == INJECTION_YEAR_CURR)[0][0])

# One threshold, computed on train and applied to every split, so the region is the same
# place in feature space everywhere.
X_train_inj, _ = feat_cache["train"][idx_inj_curr]
FLIP_THRESHOLD = float(np.quantile(X_train_inj[:, INJECTION_BAND_IDX], INJECTION_REGION_QUANTILE))
print(f"Injection region: {INJECTION_BAND} >= {FLIP_THRESHOLD:.4f} "
      f"(q{INJECTION_REGION_QUANTILE:g} of {INJECTION_YEAR_CURR} train)")


def run_flip_injection(fraction, repeat_idx):
    global FLIP_RESULTS_DF
    cache_hit = FLIP_RESULTS_DF[
        (FLIP_RESULTS_DF["injection_fraction"] == fraction)
        & (FLIP_RESULTS_DF["repeat_idx"] == repeat_idx)
        & (FLIP_RESULTS_DF["year_curr"] == INJECTION_YEAR_CURR)
    ]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    sides = {}
    n_flipped_test = 0
    for split_name in SPLIT_NAMES:
        rng = np.random.default_rng(
            [RANDOM_STATE, SPLIT_SEED[split_name], 555, int(fraction * 1000), repeat_idx]
        )
        n_class1, n_class0 = DESIGN[split_name]

        X_prev, y_prev = feat_cache[split_name][idx_inj_prev]
        idx_a = draw_side(pools[split_name][idx_inj_prev], n_class1, n_class0, rng)

        X_curr, y_curr = feat_cache[split_name][idx_inj_curr]
        region = X_curr[:, INJECTION_BAND_IDX] >= FLIP_THRESHOLD
        y_flipped, n_flipped = flip_labels_in_region(y_curr, region, fraction, rng)
        if split_name == "test":
            n_flipped_test = n_flipped

        # Pools must be re-carved: the flip moved which rows belong to which class.
        holdout_fraction = H_HOLDOUT_FRACTION if split_name == "train" else 0.0
        pool_flipped = carve_pools(y_flipped, rng, holdout_fraction=holdout_fraction)
        idx_b = draw_side(pool_flipped, n_class1, n_class0, rng)

        sides[split_name] = (X_prev[idx_a], y_prev[idx_a], X_curr[idx_b], y_flipped[idx_b])

    metrics = joint_marginal_gap(sides, HGB_PARAMS)
    if metrics is None:
        return None

    row = {
        "year_prev": INJECTION_YEAR_PREV,
        "year_curr": INJECTION_YEAR_CURR,
        "injection_fraction": float(fraction),
        "repeat_idx": int(repeat_idx),
        "n_flipped_test": int(n_flipped_test),
        "marg_test_auc": metrics["marg_test_auc"],
        "joint_test_auc": metrics["joint_test_auc"],
        "gap_auc": metrics["gap_auc"],
    }
    FLIP_RESULTS_DF = pd.concat([FLIP_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(FLIP_RESULTS_DF, FLIP_RESULTS_PATH)
    return row


flip_jobs = [(0.0, r) for r in range(N_ZERO_REPEATS)] + [(f, 0) for f in FLIP_FRACTIONS]
for fraction, repeat_idx in tqdm(flip_jobs, desc="Conditional injection sweep"):
    run_flip_injection(fraction, repeat_idx)

flip_df = FLIP_RESULTS_DF[FLIP_RESULTS_DF["year_curr"] == INJECTION_YEAR_CURR]
flip_zero = flip_df[flip_df["injection_fraction"] == 0.0]
flip_gap_floor = flip_zero["gap_auc"].mean()
flip_gap_band = elevation_band(flip_zero["gap_auc"].std())
flip_marg_floor = flip_zero["marg_test_auc"].mean()
flip_marg_band = elevation_band(flip_zero["marg_test_auc"].std())

flip_curve = flip_df.groupby("injection_fraction", as_index=False).agg(
    marg_test_auc=("marg_test_auc", "mean"),
    joint_test_auc=("joint_test_auc", "mean"),
    gap_auc=("gap_auc", "mean"),
    n_flipped_test=("n_flipped_test", "max"),
).sort_values("injection_fraction")
display(flip_curve)

plt.figure(figsize=(8.5, 4.5))
plt.plot(flip_curve["injection_fraction"], flip_curve["gap_auc"], marker="o", label="gap_auc")
plt.plot(flip_curve["injection_fraction"], flip_curve["marg_test_auc"] - flip_marg_floor,
         marker="s", label="marginal AUC (excess over its floor)")
plt.axhspan(flip_gap_floor - flip_gap_band, flip_gap_floor + flip_gap_band,
            alpha=0.15, color="gray", label="no-injection band")
plt.xlabel("Flip fraction f")
plt.ylabel("Statistic")
plt.title(f"Control A - conditional injection: {INJECTION_YEAR_PREV} vs {INJECTION_YEAR_CURR}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

gap_rank_corr = flip_curve["injection_fraction"].corr(flip_curve["gap_auc"], method="spearman")
gap_fires = flip_curve["gap_auc"].iloc[-1] > flip_gap_floor + flip_gap_band
marg_quiet = bool((flip_curve["marg_test_auc"] - flip_marg_floor).abs().max() <= flip_marg_band)

print(f"\nSpearman(flip fraction, gap_auc) = {gap_rank_corr:.3f}")
print(f"gap at f={flip_curve['injection_fraction'].iloc[-1]:g}: "
      f"{flip_curve['gap_auc'].iloc[-1]:.4f} vs band {flip_gap_floor + flip_gap_band:.4f}")
if gap_fires and gap_rank_corr >= 0.9 and marg_quiet:
    print("PASS: gap rises monotonically with conditional injection; marginal stays at its floor.")
else:
    print("WARNING: control A did not behave as designed - do not trust Section 11 until resolved.")
    print(f"  gap_fires={gap_fires}, rank_corr_ok={gap_rank_corr >= 0.9}, marg_quiet={marg_quiet}")

## 9. Control B - Covariate-Only Injection (the gap must stay quiet)

The mirror image. Reweight `year_curr`'s marginal by *selecting* rows with probability
proportional to `sigmoid(alpha * z(band))` - rows are never modified, and the selection score is a
function of X only, so **P(y|x) is preserved exactly for every x**. Raising `alpha` moves P(X)
harder while leaving the conditional alone. The marginal discriminator must climb; the gap must
not.

Why resampling rather than the band-shift perturbation used by the class-conditional notebook: a
constant added to every band of the later year is *not* a covariate-only control. Moving x while
leaving y behind gives P_B(y|x) = P_A(y|x - delta), which is itself a conditional change - it
would move the gap and prove nothing about specificity.

One design detail: resampling shrinks the `year_curr` pools, so this section computes its own
`(n_class1, n_class0)` across all alphas and both years. Its `alpha = 0` point (a plain uniform
subsample, no tilt) is repeated to give the section its own null band at that design.

In [ ]:
COVARIATE_RESULTS_COLUMNS = ["year_prev", "year_curr", "alpha", "repeat_idx",
                             "marg_test_auc", "joint_test_auc", "gap_auc"]
COVARIATE_RESULTS_DF = load_pickle_checkpoint(COVARIATE_RESULTS_PATH, COVARIATE_RESULTS_COLUMNS)


def resample_indices(split_name, alpha, repeat_idx):
    """Covariate-only row selection for year_curr in one split."""
    X_curr, _ = feat_cache[split_name][idx_inj_curr]
    rng = np.random.default_rng(
        [RANDOM_STATE, SPLIT_SEED[split_name], 777, int(alpha * 1000), repeat_idx]
    )
    return importance_resample(
        X_curr[:, INJECTION_BAND_IDX], alpha, COVARIATE_KEEP_FRACTION, rng
    )


# One design for the whole section, taken over every alpha it will visit plus the
# untouched prev year, so all sweep points stay comparable to each other.
covariate_alpha_jobs = [(0.0, r) for r in range(N_ZERO_REPEATS)] + [(a, 0) for a in COVARIATE_ALPHAS]
COVARIATE_DESIGN = {}
for split_name in SPLIT_NAMES:
    holdout_fraction = H_HOLDOUT_FRACTION if split_name == "train" else 0.0
    section_pools = [pools[split_name][idx_inj_prev]]
    for alpha, repeat_idx in covariate_alpha_jobs:
        _, y_curr = feat_cache[split_name][idx_inj_curr]
        keep = resample_indices(split_name, alpha, repeat_idx)
        section_pools.append(carve_pools(
            y_curr[keep],
            np.random.default_rng([RANDOM_STATE, SPLIT_SEED[split_name], 778]),
            holdout_fraction=holdout_fraction,
        ))
    COVARIATE_DESIGN[split_name] = design_sizes(section_pools, CLASS0_PER_CLASS1, CLASS0_CAP)

print("Section-specific design (smaller than the main design - half the rows are dropped):")
for split_name in SPLIT_NAMES:
    print(f"  {split_name}: main={DESIGN[split_name]}  covariate-section={COVARIATE_DESIGN[split_name]}")


def run_covariate_control(alpha, repeat_idx):
    global COVARIATE_RESULTS_DF
    cache_hit = COVARIATE_RESULTS_DF[
        (COVARIATE_RESULTS_DF["alpha"] == alpha)
        & (COVARIATE_RESULTS_DF["repeat_idx"] == repeat_idx)
        & (COVARIATE_RESULTS_DF["year_curr"] == INJECTION_YEAR_CURR)
    ]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    sides = {}
    for split_name in SPLIT_NAMES:
        rng = np.random.default_rng(
            [RANDOM_STATE, SPLIT_SEED[split_name], 779, int(alpha * 1000), repeat_idx]
        )
        n_class1, n_class0 = COVARIATE_DESIGN[split_name]
        holdout_fraction = H_HOLDOUT_FRACTION if split_name == "train" else 0.0

        X_prev, y_prev = feat_cache[split_name][idx_inj_prev]
        idx_a = draw_side(pools[split_name][idx_inj_prev], n_class1, n_class0, rng)

        X_curr, y_curr = feat_cache[split_name][idx_inj_curr]
        keep = resample_indices(split_name, alpha, repeat_idx)
        X_kept, y_kept = X_curr[keep], y_curr[keep]
        pool_kept = carve_pools(
            y_kept,
            np.random.default_rng([RANDOM_STATE, SPLIT_SEED[split_name], 778]),
            holdout_fraction=holdout_fraction,
        )
        idx_b = draw_side(pool_kept, n_class1, n_class0, rng)

        sides[split_name] = (X_prev[idx_a], y_prev[idx_a], X_kept[idx_b], y_kept[idx_b])

    metrics = joint_marginal_gap(sides, HGB_PARAMS)
    if metrics is None:
        return None

    row = {
        "year_prev": INJECTION_YEAR_PREV,
        "year_curr": INJECTION_YEAR_CURR,
        "alpha": float(alpha),
        "repeat_idx": int(repeat_idx),
        "marg_test_auc": metrics["marg_test_auc"],
        "joint_test_auc": metrics["joint_test_auc"],
        "gap_auc": metrics["gap_auc"],
    }
    COVARIATE_RESULTS_DF = pd.concat([COVARIATE_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(COVARIATE_RESULTS_DF, COVARIATE_RESULTS_PATH)
    return row


for alpha, repeat_idx in tqdm(covariate_alpha_jobs, desc="Covariate-only control sweep"):
    run_covariate_control(alpha, repeat_idx)

covariate_df = COVARIATE_RESULTS_DF[COVARIATE_RESULTS_DF["year_curr"] == INJECTION_YEAR_CURR]
covariate_zero = covariate_df[covariate_df["alpha"] == 0.0]
cov_gap_floor = covariate_zero["gap_auc"].mean()
cov_gap_band = elevation_band(covariate_zero["gap_auc"].std())

covariate_curve = covariate_df.groupby("alpha", as_index=False).agg(
    marg_test_auc=("marg_test_auc", "mean"),
    joint_test_auc=("joint_test_auc", "mean"),
    gap_auc=("gap_auc", "mean"),
).sort_values("alpha")
display(covariate_curve)

fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
ax1.plot(covariate_curve["alpha"], covariate_curve["marg_test_auc"], marker="s",
         color="tab:orange", label="marginal AUC")
ax1.set_xlabel("Resampling strength alpha")
ax1.set_ylabel("Marginal test AUC", color="tab:orange")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(covariate_curve["alpha"], covariate_curve["gap_auc"], marker="o",
         color="tab:blue", label="gap_auc")
ax2.axhspan(cov_gap_floor - cov_gap_band, cov_gap_floor + cov_gap_band,
            alpha=0.15, color="gray")
ax2.set_ylabel("gap_auc (shaded: no-shift band)", color="tab:blue")

fig.suptitle(f"Control B - covariate-only injection: {INJECTION_YEAR_PREV} vs {INJECTION_YEAR_CURR}")
fig.tight_layout()
plt.show()

marg_climbs = covariate_curve["marg_test_auc"].iloc[-1] > covariate_curve["marg_test_auc"].iloc[0] + 0.02
gap_quiet = bool((covariate_curve["gap_auc"] - cov_gap_floor).abs().max() <= cov_gap_band)

print(f"\nmarginal AUC: {covariate_curve['marg_test_auc'].iloc[0]:.4f} (alpha=0) -> "
      f"{covariate_curve['marg_test_auc'].iloc[-1]:.4f} (alpha={covariate_curve['alpha'].iloc[-1]:g})")
print(f"max |gap - floor| = {(covariate_curve['gap_auc'] - cov_gap_floor).abs().max():.4f} "
      f"vs band {cov_gap_band:.4f}")
if marg_climbs and gap_quiet:
    print("PASS: covariate shift moves the marginal only - the gap is specific to P(Y|X).")
else:
    print("WARNING: control B did not behave as designed - the gap may be picking up plain")
    print("         covariate shift. Do not trust Section 11 until resolved.")
    print(f"  marg_climbs={marg_climbs}, gap_quiet={gap_quiet}")

## 10. Residual-Channel Variant (secondary)

Same test, but the joint discriminator is fed `[X, y - h(X)]` instead of `[X, y]`, where `h` is a
label model fit on `year_prev`.

This is **not** an information gain: `h(X)` is a deterministic function of `X`, and `X` is already
in the feature set, so `[X, y - h(X)]` and `[X, y]` carry exactly the same information. It is a
finite-sample bet - the residual concentrates the conditional signal into one column a
depth-limited tree can split on directly, instead of making it reconstruct `h` internally. It
should agree in direction with Section 5 and may be more sensitive; where the two disagree, the
`[X, y]` number is the headline and the disagreement is the finding.

**The holdout discipline matters here.** `h` is fit on the train-split holdout carved in Section 3,
so every row it later scores is out-of-sample on *both* sides. Fitting `h` on rows that also reach
the discriminator would give the `year_prev` side optimistically small residuals and the
`year_curr` side honest ones - an asymmetry that reads as drift when there is none.
`tests/drift/test_joint.py::test_residual_channel_stays_flat_under_the_null` pins that down.

In [ ]:
RESIDUAL_RESULTS_COLUMNS = ["pair", "year_prev", "year_curr", "marg_test_auc",
                            "joint_test_auc", "gap_auc", "n_h_fit_rows", "time_sec"]
RESIDUAL_RESULTS_DF = load_pickle_checkpoint(RESIDUAL_RESULTS_PATH, RESIDUAL_RESULTS_COLUMNS)


def fit_holdout_label_model(idx_prev, rng):
    """Fit h on year_prev's train holdout - rows no discriminator ever sees."""
    pool_prev = pools["train"][idx_prev]
    holdout_idx = np.concatenate([pool_prev.holdout_class1, pool_prev.holdout_class0])
    if len(holdout_idx) > H_FIT_CAP:
        holdout_idx = rng.choice(holdout_idx, size=H_FIT_CAP, replace=False)

    X_train, y_train = feat_cache["train"][idx_prev]
    return fit_label_model(X_train[holdout_idx], y_train[holdout_idx], HGB_PARAMS), len(holdout_idx)


def run_residual_pair(idx_prev, idx_curr):
    global RESIDUAL_RESULTS_DF
    year_prev = int(year_values[idx_prev])
    year_curr = int(year_values[idx_curr])
    pair_name = f"{year_prev} vs {year_curr}"

    cache_hit = RESIDUAL_RESULTS_DF[RESIDUAL_RESULTS_DF["pair"] == pair_name]
    if len(cache_hit) > 0:
        return cache_hit.iloc[-1].to_dict()

    start = time.time()
    rng = np.random.default_rng([RANDOM_STATE, 313, idx_prev, idx_curr])
    label_model, n_h_fit_rows = fit_holdout_label_model(idx_prev, rng)

    # Same seed_tag as run_pair, so this runs on exactly the rows Section 5 used.
    sides = draw_pair_sides(idx_prev, idx_curr, seed_tag=(idx_prev, idx_curr))
    extra_channel = {
        split_name: (
            residual_channel(label_model, X_a, y_a),
            residual_channel(label_model, X_b, y_b),
        )
        for split_name, (X_a, y_a, X_b, y_b) in sides.items()
    }

    metrics = joint_marginal_gap(sides, HGB_PARAMS, extra_channel=extra_channel)
    if metrics is None:
        return None

    row = {
        "pair": pair_name,
        "year_prev": year_prev,
        "year_curr": year_curr,
        "marg_test_auc": metrics["marg_test_auc"],
        "joint_test_auc": metrics["joint_test_auc"],
        "gap_auc": metrics["gap_auc"],
        "n_h_fit_rows": int(n_h_fit_rows),
        "time_sec": float(time.time() - start),
    }
    RESIDUAL_RESULTS_DF = pd.concat([RESIDUAL_RESULTS_DF, pd.DataFrame([row])], ignore_index=True)
    save_pickle_checkpoint(RESIDUAL_RESULTS_DF, RESIDUAL_RESULTS_PATH)
    return row


residual_start = time.time()
for idx_prev, idx_curr in tqdm(adjacent_idx_pairs, desc="Residual-channel adjacent pairs"):
    run_residual_pair(idx_prev, idx_curr)
print(f"Residual section done in {time.time() - residual_start:.1f}s")

residual_compare = (
    adjacent_rows(PAIR_RESULTS_DF)[["pair", "year_curr", "gap_auc"]]
    .rename(columns={"gap_auc": "gap_auc_label"})
    .merge(
        RESIDUAL_RESULTS_DF[["pair", "gap_auc"]].rename(columns={"gap_auc": "gap_auc_residual"}),
        on="pair", how="left",
    )
    .sort_values("year_curr")
    .reset_index(drop=True)
)
display(residual_compare)

x = np.arange(len(residual_compare))
plt.figure(figsize=(9, 4.5))
plt.bar(x - 0.2, residual_compare["gap_auc_label"], 0.4, label="[X, y] channel")
plt.bar(x + 0.2, residual_compare["gap_auc_residual"], 0.4, label="[X, y - h(X)] channel")
plt.xticks(x, residual_compare["pair"], rotation=45, ha="right")
plt.ylabel("gap_auc")
plt.title("Label channel vs residual channel (adjacent pairs)")
plt.grid(alpha=0.3, axis="y")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Headline Table

Each pair's `gap_auc` and `marg_test_auc` read against year_curr's own same-year null from
Section 7. "Elevated" means more than `NULL_ELEVATED_Z` null-std above the null mean **and** at
least `MIN_AUC_BAND` above it in absolute terms - a null estimated from a handful of repeats can
come out arbitrarily tight by luck, and an AUC difference below that floor is not actionable
however significant it computes.

| gap | marginal | reading |
|---|---|---|
| null | null | no drift beyond prior shift (Section 1 is then the whole story) |
| null | elevated | covariate shift only - X moved, P(Y\|X) held |
| elevated | null | **real concept drift** - the relationship moved, X's marginal did not |
| elevated | elevated | both - X moved *and* the relationship moved |

The two numbers answer different questions and both belong in any report of a pair. `gap_auc`
alone ranks pairs by how much of their drift is conditional; it does not say whether that pair
drifted at all.

In [ ]:
null_lookup = null_summary_df.set_index("year")

gap_rows = []
for _, r in PAIR_RESULTS_DF.iterrows():
    if r["year_curr"] not in null_lookup.index:
        continue
    null_row = null_lookup.loc[r["year_curr"]]

    gap_excess = r["gap_auc"] - null_row["gap_null_mean"]
    marg_excess = r["marg_test_auc"] - null_row["marg_null_mean"]
    gap_elevated = bool(gap_excess > elevation_band(null_row["gap_null_std"]))
    marg_elevated = bool(marg_excess > elevation_band(null_row["marg_null_std"]))

    prior_prev = prior_rate_df[(prior_rate_df.year == r["year_prev"]) & (prior_rate_df.split == "test")]["positive_rate"]
    prior_curr = prior_rate_df[(prior_rate_df.year == r["year_curr"]) & (prior_rate_df.split == "test")]["positive_rate"]

    gap_rows.append({
        "pair": r["pair"],
        "year_prev": int(r["year_prev"]),
        "year_curr": int(r["year_curr"]),
        "prior_rate_prev": float(prior_prev.iloc[0]) if len(prior_prev) else np.nan,
        "prior_rate_curr": float(prior_curr.iloc[0]) if len(prior_curr) else np.nan,
        "marg_auc": float(r["marg_test_auc"]),
        "joint_auc": float(r["joint_test_auc"]),
        "gap_auc": float(r["gap_auc"]),
        "gap_null_mean": float(null_row["gap_null_mean"]),
        "gap_excess": float(gap_excess),
        "gap_elevated": gap_elevated,
        "marg_excess": float(marg_excess),
        "marg_elevated": marg_elevated,
        "gap_logloss_reduction": float(r["gap_logloss_reduction"]),
        "reading": classify_gap(gap_elevated, marg_elevated),
    })

gap_df = pd.DataFrame(gap_rows).sort_values(["year_curr", "year_prev"]).reset_index(drop=True)
gap_df.to_csv(GAP_TABLE_PATH, index=False)
print(f"Wrote {GAP_TABLE_PATH}")

adjacent_gap_df = gap_df[gap_df["year_curr"] - gap_df["year_prev"] == 1].reset_index(drop=True)
display(adjacent_gap_df[["pair", "prior_rate_prev", "prior_rate_curr", "marg_auc",
                         "joint_auc", "gap_auc", "gap_excess", "reading"]])

x = np.arange(len(adjacent_gap_df))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, adjacent_gap_df["gap_auc"], 0.55, label="gap_auc", color="tab:blue")
ax.plot(x, adjacent_gap_df["gap_null_mean"], "k_", markersize=28, label="same-year null mean")
for i, row in enumerate(adjacent_gap_df.itertuples()):
    ax.annotate(f"marg {row.marg_auc:.3f}", (i, 0.0), ha="center", va="top", fontsize=8, color="gray")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(adjacent_gap_df["pair"], rotation=45, ha="right")
ax.set_ylabel("gap_auc  (joint - marginal)")
ax.set_title("Real-drift gap per adjacent-year pair (gray: that pair's marginal AUC)")
ax.grid(alpha=0.3, axis="y")
ax.legend()
plt.tight_layout()
plt.show()

if len(gap_df) > len(adjacent_gap_df):
    heat = gap_df.pivot(index="year_prev", columns="year_curr", values="gap_auc")
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(heat.values, cmap="viridis")
    ax.set_xticks(range(len(heat.columns)), heat.columns)
    ax.set_yticks(range(len(heat.index)), heat.index)
    ax.set_xlabel("year_curr")
    ax.set_ylabel("year_prev")
    ax.set_title("gap_auc, every year against every earlier year")
    for i in range(len(heat.index)):
        for j in range(len(heat.columns)):
            if not np.isnan(heat.values[i, j]):
                ax.text(j, i, f"{heat.values[i, j]:.3f}", ha="center", va="center",
                        color="white", fontsize=8)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## 12. Cross-Check Against the Class-Conditional Notebook

`concept_drift_detection-class-conditional-hgb.ipynb` measures the same underlying quantity
through a different route: two X-only class-conditional discriminators, with
`concept_asymmetry = (class1_auc - null1) - (class0_auc - null0)` as its summary. Under matched
priors the two statistics fire on the same condition - class-asymmetric movement of P(x|y) - so
they should rank pairs similarly.

Agreement is corroboration. Disagreement is a finding worth chasing, not a bug to paper over: the
class-conditional route caps class-0 for compute and uses the natural class mix, while this
notebook uses a near-balanced matched design, so the two see different slices of the same data.

In [ ]:
lift_path = EXPERIMENTS_DIR / "concept_drift_class_conditional_hgb_all_prev" / "lift_table.csv"

if not lift_path.exists():
    print(f"Class-conditional lift table not found at {lift_path} - skipping cross-check.")
    print("Run concept_drift_detection-class-conditional-hgb.ipynb first to enable it.")
else:
    lift_df = pd.read_csv(lift_path)
    cross_df = gap_df.merge(
        lift_df[["year_prev", "year_curr", "class1_auc", "class0_auc", "concept_asymmetry", "reading"]]
        .rename(columns={"reading": "class_conditional_reading"}),
        on=["year_prev", "year_curr"],
        how="inner",
    )

    if len(cross_df) == 0:
        print("No overlapping pairs between the two notebooks.")
    else:
        display(cross_df[["pair", "gap_auc", "concept_asymmetry",
                          "reading", "class_conditional_reading"]])

        rho = cross_df["gap_auc"].corr(cross_df["concept_asymmetry"], method="spearman")
        print(f"\nSpearman rho(gap_auc, concept_asymmetry) = {rho:.3f} (n={len(cross_df)})")

        agree = (
            cross_df["reading"].str.contains("real concept drift")
            == cross_df["class_conditional_reading"].str.contains("concept drift")
        )
        print(f"Verdict agreement: {int(agree.sum())}/{len(cross_df)} pairs")
        if not agree.all():
            print("\nPairs where the two routes disagree:")
            display(cross_df[~agree][["pair", "gap_auc", "concept_asymmetry",
                                      "reading", "class_conditional_reading"]])

        plt.figure(figsize=(6, 5))
        plt.scatter(cross_df["concept_asymmetry"], cross_df["gap_auc"])
        for _, r in cross_df.iterrows():
            plt.annotate(r["pair"], (r["concept_asymmetry"], r["gap_auc"]), fontsize=7)
        plt.xlabel("concept_asymmetry (class-conditional route)")
        plt.ylabel("gap_auc (joint route)")
        plt.title("Two routes to the same quantity")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## 13. External Anchor (secondary, caveated)

Correlates `gap_auc` against next-year model degradation from the unified-eval baseline, merged on
`model_year == year_prev` and `eval_year == year_curr`.

Real concept drift is the kind that should actually hurt a deployed model - covariate shift alone
can leave a well-specified model's performance intact - so a negative correlation between
`gap_auc` and next-year performance is the outcome the method predicts.

**Caveats, stated rather than left implicit:**
- n is around 5 adjacent pairs. Read this as directional only; it cannot confirm or refute the method.
- The baseline model conditions on `s2_bands + dem + ndvi + ndwi + nbr` and their temporal deltas,
  while this discriminator sees 7 raw S2 bands only. A weak correlation is ambiguous between "no
  concept drift" and "the richer feature set absorbs what raw bands show".
- Sections 8 and 9 are the actual validation of the statistic. This section is context.

In [ ]:
baseline_path = (
    PROJECT_ROOT / "experiments" / "evaluation" / "eval_outputs" / "unified_eval"
    / "baseline" / "baseline_next_year.csv"
)

if not baseline_path.exists():
    print(f"Baseline eval file not found at {baseline_path} - skipping external anchor.")
else:
    baseline_df = pd.read_csv(baseline_path)
    anchor_df = adjacent_gap_df.merge(
        baseline_df[["model_year", "eval_year", "recall", "f1_score", "roc_auc"]],
        left_on=["year_prev", "year_curr"],
        right_on=["model_year", "eval_year"],
        how="left",
    )
    display(anchor_df[["pair", "gap_auc", "marg_auc", "recall", "f1_score", "roc_auc"]])

    n_valid = int(anchor_df["recall"].notna().sum())
    if n_valid >= 3:
        rho_gap = anchor_df["gap_auc"].corr(anchor_df["recall"], method="spearman")
        rho_marg = anchor_df["marg_auc"].corr(anchor_df["recall"], method="spearman")
        print(f"\nSpearman rho(gap_auc,  next-year recall) = {rho_gap:.3f} (n={n_valid})")
        print(f"Spearman rho(marg_auc, next-year recall) = {rho_marg:.3f} (n={n_valid})")
        print("Directional only - see the caveats above.")

    plt.figure(figsize=(6, 5))
    plt.scatter(anchor_df["gap_auc"], anchor_df["recall"])
    for _, r in anchor_df.iterrows():
        plt.annotate(r["pair"], (r["gap_auc"], r["recall"]), fontsize=8)
    plt.xlabel("gap_auc")
    plt.ylabel("Next-year model recall (disturbance class)")
    plt.title(f"Real-drift gap vs next-year degradation (n={n_valid} - directional only)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Interpretation Caveats

- **Label provenance.** A P(Y|X) shift can mean "the spectral signature of disturbance changed" or
  "the labeling methodology changed". Nothing in this data separates the two. Any pair whose
  `gap_auc` is elevated *and* whose prior rate jumps (Section 1) should be reported with both
  explanations on the table.
- **Two numbers, not one.** `gap_auc` and `marg_auc` answer different questions. The Section 11
  reading is built from both; `gap_auc` alone ranks pairs but does not say whether a pair drifted.
- **Paired rows.** The same physical pixels appear on both sides of a pair, possibly under
  different labels across years, so rows are not independent draws and nominal AUC confidence
  intervals would be too narrow. The same-year null in Section 7 absorbs this empirically rather
  than relying on a corrected-variance formula.
- **The design bounds what can be seen.** Matching priors and fixing row counts costs data: the
  design is set by the scarcest year's class-1 count (Section 3). A drift confined to a small
  region of feature space may not clear the null band at these row counts. Absence of an elevated
  gap is not proof of a stable P(Y|X).
- **Imputation.** Features come from `src.drift.features`, which fills S2 NaNs with a per-pixel
  mean across *all* years - the same known temporal-leakage caveat carried by every other drift
  notebook here. It is shared rather than fixed so numbers stay comparable across them.
- **The controls are per-pair, not global.** Sections 8 and 9 validate the statistic on one year
  pair. They establish that the machinery separates conditional from covariate shift on this data;
  they do not certify every pair in Section 11 individually.